# 02 - Treat Service List

## Purpose
Count aged care facilities (residential + home care) per SA3 per year, and track
how the supply landscape has changed over 2019–2025.

## Input
- `data/raw/service_list/` — 7 annual snapshots (2019–2025 XLSX)

## Output
- `data/clean/service_supply_by_sa3.csv` — SA3 × year with facility counts, licensed places, and org type breakdown

## Key columns
- `n_facilities`, `n_residential`, `n_homecare`
- `residential_places`, `homecare_places`
- `n_nfp` — not-for-profit (Charitable + Religious + Community Based)
- `n_government` — State / Local / Territory Government
- `n_private` — Private Incorporated Body + Publicly Listed Company

## Key context
- Only 2023–2025 files include `2016 SA3 Code` directly; 2019–2022 use postcode + ACPR lookup
- `Residential Places` = licensed beds (supply capacity, not utilisation)
- A SA3 with high `n_private` share but low quality = market failure signal

In [1]:
import pandas as pd
import os

RAW = '../../data/raw/service_list'
OUT = '../../data/clean/service_supply_by_sa3.csv'

In [2]:
# =============================================================================
# STEP 1: Build postcode â†’ SA3 lookup from 2023â€“2025 service lists
# =============================================================================
# 2019â€“2022 files don't have SA3 codes â€” only postcode + ACPR.
# We derive the mapping internally from 2023â€“2025 (which have postcode, ACPR, and SA3).
#
# Two-tier lookup:
#   Tier 1 (unambiguous): postcode â†’ 1 SA3 â†’ use directly
#   Tier 2 (ambiguous):   postcode â†’ 2+ SA3 â†’ use (postcode + ACPR) to pick the right one
#
# ACPR values are identical across all years (confirmed: 67/67 exact match 2019â€“2025).

def read_service_list(filepath):
    probe = pd.read_excel(filepath, header=None, nrows=6)
    hdr_row = next(i for i, r in probe.iterrows() if 'Service Name' in r.values)
    return pd.read_excel(filepath, header=hdr_row)

files = sorted([f for f in os.listdir(RAW) if not f.startswith('~')])

POSTCODE_COL = {2023: 'Physical Post Code', 2024: 'Postal Code', 2025: 'Physical Post Code'}

mapping_frames = []
for year in [2023, 2024, 2025]:
    fname = next(f for f in files if str(year) in f)
    df = read_service_list(f'{RAW}/{fname}')
    sa3_col      = next(c for c in df.columns if 'SA3 Code' in str(c))
    sa3_name_col = next(c for c in df.columns if 'SA3 Name' in str(c))
    acpr_col     = next(c for c in df.columns if 'ACPR' in str(c) or 'Planning Region' in str(c))
    tmp = df[[POSTCODE_COL[year], acpr_col, sa3_col, sa3_name_col]].dropna()
    tmp.columns = ['postcode', 'acpr', 'sa3_code', 'sa3_name']
    tmp['postcode'] = tmp['postcode'].astype(str).str.strip().str.split('.').str[0]
    mapping_frames.append(tmp)

raw_mapping = pd.concat(mapping_frames, ignore_index=True)

# Tier 1: postcodes that map to exactly 1 SA3 â€” use directly
pc_sa3_counts = raw_mapping.groupby('postcode')['sa3_code'].nunique()
unambiguous   = set(pc_sa3_counts[pc_sa3_counts == 1].index)
ambiguous     = set(pc_sa3_counts[pc_sa3_counts > 1].index)

postcode_sa3_simple = (
    raw_mapping[raw_mapping['postcode'].isin(unambiguous)]
    .drop_duplicates('postcode')[['postcode', 'sa3_code', 'sa3_name']]
)

# Tier 2: ambiguous postcodes â€” use (postcode + ACPR) to pick the right SA3
# Take the most frequent (postcode, acpr) â†’ sa3 combination
postcode_acpr_sa3 = (
    raw_mapping[raw_mapping['postcode'].isin(ambiguous)]
    .groupby(['postcode', 'acpr', 'sa3_code', 'sa3_name'])
    .size().reset_index(name='freq')
    .sort_values('freq', ascending=False)
    .drop_duplicates(['postcode', 'acpr'])
    [['postcode', 'acpr', 'sa3_code', 'sa3_name']]
)

print(f'Tier 1 (unambiguous postcodes):      {len(postcode_sa3_simple):,}')
print(f'Tier 2 (postcode+ACPR combos):       {len(postcode_acpr_sa3):,}  (from {len(ambiguous)} ambiguous postcodes)')
print(f'Total postcodes covered:             {len(unambiguous) + len(ambiguous):,}')

Tier 1 (unambiguous postcodes):      1,361
Tier 2 (postcode+ACPR combos):       222  (from 87 ambiguous postcodes)
Total postcodes covered:             1,448


In [3]:
# =============================================================================
# STEP 2: Process ALL years (2019–2025) → SA3-level supply
# =============================================================================

sa3_frames = []
year_file_map = {yr: next(f for f in files if str(yr) in f) for yr in range(2019, 2026)}

NFP_TYPES  = {'charitable', 'religious', 'community based', 'religious/charitable'}
GOVT_TYPES = {'state government', 'local government', 'territory government'}
PRIV_TYPES = {'private incorporated body', 'publicly listed company'}

for year, fname in sorted(year_file_map.items()):
    df = read_service_list(f'{RAW}/{fname}')
    print(f'\n{year} — {fname}')

    has_sa3 = any('SA3 Code' in str(c) for c in df.columns)

    if has_sa3:
        sa3_col      = next(c for c in df.columns if 'SA3 Code' in str(c))
        sa3_name_col = next(c for c in df.columns if 'SA3 Name' in str(c))
        df = df.rename(columns={sa3_col: 'sa3_code', sa3_name_col: 'sa3_name'})
    else:
        acpr_col = next(c for c in df.columns if 'ACPR' in str(c) or 'Planning Region' in str(c))
        df['postcode'] = df['Physical Address Post Code'].astype(str).str.strip().str.split('.').str[0]
        df = df.rename(columns={acpr_col: 'acpr'})

        df = df.merge(postcode_sa3_simple, on='postcode', how='left')
        mask = df['sa3_code'].isna()
        if mask.sum() > 0:
            fill = df.loc[mask, ['postcode', 'acpr']].merge(
                postcode_acpr_sa3, on=['postcode', 'acpr'], how='left'
            )
            df.loc[mask, 'sa3_code'] = fill['sa3_code'].values
            df.loc[mask, 'sa3_name'] = fill['sa3_name'].values

        n_mapped = df['sa3_code'].notna().sum()
        print(f'  SA3 match: {n_mapped}/{len(df)} ({n_mapped/len(df)*100:.1f}%)'
              f'  [tier2 resolved: {mask.sum() - df["sa3_code"].isna().sum()}]')

    care_col = next((c for c in df.columns if 'Care Type' in str(c)), None)
    if not care_col:
        print('  WARNING: no Care Type column, skipping')
        continue

    df['is_residential'] = df[care_col].str.contains(
        'Residential|Multi-Purpose', na=False, case=False).astype(int)
    df['is_homecare'] = df[care_col].str.contains(
        'Home Care', na=False, case=False).astype(int)

    for col in ['Residential Places', 'Home Care Places']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)
        else:
            df[col] = 0

    # Organisation type flags
    org_col = next((c for c in df.columns if 'Organisation' in str(c)), None)
    if org_col:
        org = df[org_col].astype(str).str.strip().str.lower()
        df['_nfp']     = org.isin(NFP_TYPES).astype(int)
        df['_govt']    = org.isin(GOVT_TYPES).astype(int)
        df['_private'] = org.isin(PRIV_TYPES).astype(int)
    else:
        df['_nfp'] = df['_govt'] = df['_private'] = 0

    agg = (
        df.dropna(subset=['sa3_code'])
        .groupby(['sa3_code', 'sa3_name'])
        .agg(
            n_facilities       = ('Service Name',       'count'),
            n_residential      = ('is_residential',     'sum'),
            n_homecare         = ('is_homecare',        'sum'),
            residential_places = ('Residential Places', 'sum'),
            homecare_places    = ('Home Care Places',   'sum'),
            n_nfp              = ('_nfp',               'sum'),
            n_government       = ('_govt',              'sum'),
            n_private          = ('_private',           'sum'),
        )
        .reset_index()
    )
    agg['year'] = year
    sa3_frames.append(agg)
    print(f'  SA3 regions: {len(agg)} | residential: {int(agg["n_residential"].sum())} | '
          f'nfp: {int(agg["n_nfp"].sum())} | govt: {int(agg["n_government"].sum())} | private: {int(agg["n_private"].sum())}')

supply_sa3 = pd.concat(sa3_frames, ignore_index=True)
print(f'\nCombined supply_sa3: {supply_sa3.shape}')


2019 — Australia-30-June-2019-v2-1.xlsx
  SA3 match: 5758/5796 (99.3%)  [tier2 resolved: 730]
  SA3 regions: 328 | residential: 2877 | nfp: 3538 | govt: 712 | private: 1508



2020 — Australia-at-30-June-2020.xlsx
  SA3 match: 5751/5776 (99.6%)  [tier2 resolved: 736]
  SA3 regions: 328 | residential: 2888 | nfp: 3512 | govt: 707 | private: 1532



2021 — Australia_30-June-2021.xlsx
  SA3 match: 5734/5756 (99.6%)  [tier2 resolved: 738]
  SA3 regions: 329 | residential: 2873 | nfp: 3454 | govt: 703 | private: 1577



2022 — Australia_30-June-2022.xlsx
  SA3 match: 5522/5534 (99.8%)  [tier2 resolved: 690]
  SA3 regions: 329 | residential: 2841 | nfp: 3289 | govt: 670 | private: 1563



2023 — Australia-Service-List-2023.xlsx
  SA3 regions: 331 | residential: 2821 | nfp: 3308 | govt: 644 | private: 1571



2024 — Service-List-30-Jun-2024-Australia.xlsx
  SA3 regions: 331 | residential: 2800 | nfp: 3185 | govt: 627 | private: 1602



2025 — Service-List-2025-Australia_300126.xlsx
  SA3 regions: 331 | residential: 2773 | nfp: 3193 | govt: 602 | private: 1583

Combined supply_sa3: (2307, 11)


In [4]:
# =============================================================================
# STEP 3: Sanity checks
# =============================================================================

print('=== SA3 regions with ZERO residential facilities (by year) ===')
no_res = supply_sa3[supply_sa3['n_residential'] == 0].groupby('year').size()
print(no_res)

print('\n=== National facility totals by year ===')
by_year = supply_sa3.groupby('year')[['n_residential', 'n_homecare',
                                       'residential_places', 'homecare_places']].sum()
print(by_year.to_string())

print('\n=== Organisation type breakdown by year ===')
org_year = supply_sa3.groupby('year')[['n_nfp', 'n_government', 'n_private']].sum()
org_year['total'] = org_year.sum(axis=1)
org_year['pct_private'] = (org_year['n_private'] / org_year['total'] * 100).round(1)
print(org_year.to_string())
print('>> Rising pct_private = market consolidation, potential quality risk')

print('\n=== Top 10 SA3 by private facility share (2024) ===')
yr2024 = supply_sa3[supply_sa3['year'] == 2024].copy()
yr2024['pct_private'] = yr2024['n_private'] / yr2024['n_facilities'] * 100
top_priv = (yr2024[yr2024['n_facilities'] >= 5]
            .sort_values('pct_private', ascending=False)
            .head(10)[['sa3_name', 'n_facilities', 'n_private', 'n_nfp', 'n_government', 'pct_private']])
print(top_priv.to_string(index=False))

print('\n=== SA3 regions that LOST residential facilities 2023→2025 ===')
pivot = supply_sa3.pivot_table(
    index=['sa3_code', 'sa3_name'], columns='year', values='n_residential'
).reset_index()
if 2023 in pivot.columns and 2025 in pivot.columns:
    pivot['change'] = pivot[2025] - pivot[2023]
    lost = pivot[pivot['change'] < 0].sort_values('change')
    print(f'  {len(lost)} SA3 regions lost at least one residential facility')
    print(lost[['sa3_name', 2023, 2025, 'change']].head(15).to_string(index=False))

=== SA3 regions with ZERO residential facilities (by year) ===
year
2019    4
2020    4
2021    4
2022    4
2023    5
2024    5
2025    5
dtype: int64

=== National facility totals by year ===
      n_residential  n_homecare  residential_places  homecare_places
year                                                                
2019           2877        2675            215989.0           1086.0
2020           2888        2639            220280.0           1256.0
2021           2873        2642            222297.0           1327.0
2022           2841        2424            223247.0           1332.0
2023           2821        2448            225216.0           1391.0
2024           2800        2364            227465.0           1683.0
2025           2773        2363            228259.0           1865.0

=== Organisation type breakdown by year ===
      n_nfp  n_government  n_private  total  pct_private
year                                                    
2019   3538           712  

In [5]:
# =============================================================================
# STEP 4: Save
# =============================================================================

supply_sa3['sa3_code'] = supply_sa3['sa3_code'].astype(str).str.strip()
supply_sa3.to_csv(OUT, index=False)

print(f'Saved: {supply_sa3.shape[0]:,} rows Ã— {supply_sa3.shape[1]} columns â†’ {OUT}')
print(f'Years: {sorted(supply_sa3["year"].unique())}')
print(f'SA3 regions (unique): {supply_sa3["sa3_code"].nunique()}')
print('\nColumns:', supply_sa3.columns.tolist())
print('\nSample:')
print(supply_sa3[supply_sa3['year'] == 2025].head(5).to_string(index=False))

Saved: 2,307 rows Ã— 11 columns â†’ ../../data/clean/service_supply_by_sa3.csv
Years: [np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]
SA3 regions (unique): 331

Columns: ['sa3_code', 'sa3_name', 'n_facilities', 'n_residential', 'n_homecare', 'residential_places', 'homecare_places', 'n_nfp', 'n_government', 'n_private', 'year']

Sample:
sa3_code            sa3_name  n_facilities  n_residential  n_homecare  residential_places  homecare_places  n_nfp  n_government  n_private  year
 10102.0          Queanbeyan             6              4           2               364.0              2.0      4             1          1  2025
 10103.0     Snowy Mountains             6              4           1                97.0              4.0      2             4          0  2025
 10104.0         South Coast            25             13          12              1060.0              0.0     19             1          5  2025
 10105.0 Goulburn